<a href="https://colab.research.google.com/github/AzadMahmud/dl-lab/blob/main/assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:


import os
import glob
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import umap


# ------------------ Config (small + safe) ------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG = 160
BATCH = 16
RESULTS = "/content/results"
os.makedirs(RESULTS, exist_ok=True)

FACE_TEST_DIR = "/content/face-test-images"
FLOWER_TEST_DIR = "/content/flower_test_images"

# Keep dataset sizes intentionally small for Colab RAM safety
LFW_MAX = 1200
CIFAR_TRAIN_MAX = 8000
CIFAR_VAL_MAX = 2000

# For PCA/tSNE/UMAP speed+RAM
EMBED_MAX = 1000


# ------------------ Utility ------------------
def preprocess(img):
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, (IMG, IMG))
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img


def make_ds(x, y=None, train=False):
    if y is None:
        ds = tf.data.Dataset.from_tensor_slices(x)
        ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, y))
        ds = ds.map(
            lambda a, b: (preprocess(a), b), num_parallel_calls=tf.data.AUTOTUNE
        )
    if train:
        ds = ds.shuffle(min(len(x), 4000), seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)


def subsample_xy(x, y, n):
    if n is None or len(x) <= n:
        return x, y
    idx = np.random.choice(len(x), n, replace=False)
    return x[idx], y[idx]


def build_model(num_classes):
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG, IMG, 3),
    )
    gap = tf.keras.layers.GlobalAveragePooling2D(name="gap")(base.output)
    out = tf.keras.layers.Dense(num_classes, activation="softmax", name="cls")(gap)
    model = tf.keras.Model(base.input, out)
    return model, base


def feature_model(model):
    return tf.keras.Model(model.input, model.get_layer("gap").output)


def extract_features(fm, x):
    return fm.predict(make_ds(x), verbose=0)


def load_test_images(folder):
    if not os.path.exists(folder):
        return None, None
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
    paths = []
    for e in exts:
        paths += glob.glob(os.path.join(folder, "**", e), recursive=True)
    paths = sorted(paths)
    if len(paths) == 0:
        return None, None

    imgs = []
    for p in paths:
        im = tf.keras.utils.load_img(p)
        arr = tf.keras.utils.img_to_array(im)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        imgs.append(arr)

    # test sets are small, safe to resize here
    resized = [tf.image.resize(a, (IMG, IMG)).numpy().astype(np.uint8) for a in imgs]
    return np.stack(resized, axis=0), paths


def reduce2d(feat):
    z = StandardScaler().fit_transform(feat)
    out = {}
    out["pca"] = PCA(n_components=2, random_state=SEED).fit_transform(z)
    perp = max(5, min(25, (len(z) - 1) // 3))
    out["tsne"] = TSNE(
        n_components=2,
        random_state=SEED,
        init="pca",
        learning_rate="auto",
        perplexity=perp,
    ).fit_transform(z)
    out["umap"] = umap.UMAP(
        n_components=2, n_neighbors=12, min_dist=0.1, random_state=SEED
    ).fit_transform(z)
    return out


def plot_before_after(red_b, red_a, labels, prefix):
    for k in ["pca", "tsne", "umap"]:
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].scatter(
            red_b[k][:, 0], red_b[k][:, 1], c=labels, s=12, cmap="tab20", alpha=0.85
        )
        ax[0].set_title(f"{k.upper()} before")
        ax[0].grid(alpha=0.2)
        ax[1].scatter(
            red_a[k][:, 0], red_a[k][:, 1], c=labels, s=12, cmap="tab20", alpha=0.85
        )
        ax[1].set_title(f"{k.upper()} after")
        ax[1].grid(alpha=0.2)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS, f"{prefix}_{k}_before_after.png"), dpi=180)
        plt.close()


def run_task(name, xtr, ytr, xva, yva, xtest=None, test_paths=None):
    print(f"\n===== {name} =====")
    ncls = len(np.unique(ytr))
    model, base = build_model(ncls)

    # Before FT
    fm_before = feature_model(model)
    ftr_b = extract_features(fm_before, xtr)
    fva_b = extract_features(fm_before, xva)

    # Tiny fine-tuning (minimal)
    ds_tr = make_ds(xtr, ytr, train=True)
    ds_va = make_ds(xva, yva, train=False)

    base.trainable = False
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    model.fit(ds_tr, validation_data=ds_va, epochs=1, verbose=1)

    base.trainable = True
    for l in base.layers[:-20]:
        l.trainable = False
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    model.fit(ds_tr, validation_data=ds_va, epochs=1, verbose=1)

    # After FT
    fm_after = feature_model(model)
    ftr_a = extract_features(fm_after, xtr)
    fva_a = extract_features(fm_after, xva)

    # quick quantitative comparison (kNN on features)
    knn_b = KNeighborsClassifier(n_neighbors=3, metric="cosine").fit(ftr_b, ytr)
    knn_a = KNeighborsClassifier(n_neighbors=3, metric="cosine").fit(ftr_a, ytr)
    acc_b = accuracy_score(yva, knn_b.predict(fva_b))
    acc_a = accuracy_score(yva, knn_a.predict(fva_a))
    print(f"kNN val acc before={acc_b:.4f}, after={acc_a:.4f}")

    # embeddings on validation subset only
    x = np.arange(len(fva_b))
    if len(x) > EMBED_MAX:
        x = np.random.choice(x, EMBED_MAX, replace=False)
    red_b = reduce2d(fva_b[x])
    red_a = reduce2d(fva_a[x])
    plot_before_after(red_b, red_a, yva[x], f"{name.lower()}_val")

    # own test set (colored by predicted labels)
    if xtest is not None and len(xtest) > 0:
        ft_b = extract_features(fm_before, xtest)
        ft_a = extract_features(fm_after, xtest)
        ytest_pred = knn_a.predict(ft_a)
        red_tb = reduce2d(ft_b)
        red_ta = reduce2d(ft_a)
        plot_before_after(red_tb, red_ta, ytest_pred, f"{name.lower()}_test")

        if test_paths is not None:
            import pandas as pd

            pd.DataFrame(
                {
                    "file": [os.path.basename(p) for p in test_paths],
                    "pred_after": ytest_pred,
                }
            ).to_csv(
                os.path.join(RESULTS, f"{name.lower()}_test_predictions.csv"),
                index=False,
            )

    return {
        "task": name,
        "knn_val_acc_before": float(acc_b),
        "knn_val_acc_after": float(acc_a),
    }


def main():
    print("TensorFlow:", tf.__version__)

    # ---------- Face task (LFW) ----------
    print("Loading LFW...")
    lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=True)
    x_face = lfw.images.astype(np.uint8)
    y_face = lfw.target.astype(np.int64)
    x_face, y_face = subsample_xy(x_face, y_face, LFW_MAX)
    xft, xfv, yft, yfv = train_test_split(
        x_face, y_face, test_size=0.2, random_state=SEED, stratify=y_face
    )
    print("Face train/val:", xft.shape, xfv.shape)

    # ---------- Flower task (CIFAR-10 train/val) ----------
    print("Loading CIFAR-10...")
    (xtr, ytr), (xva, yva) = tf.keras.datasets.cifar10.load_data()
    ytr = ytr.squeeze().astype(np.int64)
    yva = yva.squeeze().astype(np.int64)
    xtr, ytr = subsample_xy(xtr, ytr, CIFAR_TRAIN_MAX)
    xva, yva = subsample_xy(xva, yva, CIFAR_VAL_MAX)
    print("CIFAR train/val:", xtr.shape, xva.shape)

    # ---------- Own test images ----------
    x_face_test, face_paths = load_test_images(FACE_TEST_DIR)
    x_flower_test, flower_paths = load_test_images(FLOWER_TEST_DIR)
    print("Face test:", None if x_face_test is None else x_face_test.shape)
    print("Flower test:", None if x_flower_test is None else x_flower_test.shape)

    m1 = run_task("Face", xft, yft, xfv, yfv, x_face_test, face_paths)
    m2 = run_task("Flowers", xtr, ytr, xva, yva, x_flower_test, flower_paths)

    metrics = [m1, m2]
    with open(os.path.join(RESULTS, "summary_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    import pandas as pd

    pd.DataFrame(metrics).to_csv(
        os.path.join(RESULTS, "summary_metrics.csv"), index=False
    )

    print("\nSaved files in", RESULTS)
    print(sorted(os.listdir(RESULTS)))


if __name__ == "__main__":
    main()

TensorFlow: 2.19.0
Loading LFW...
Face train/val: (960, 62, 47, 3) (240, 62, 47, 3)
Loading CIFAR-10...
CIFAR train/val: (8000, 32, 32, 3) (2000, 32, 32, 3)
Face test: None
Flower test: None

===== Face =====
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
60/60 ━━━━━━━━━━━━━━━━━━━━ 16s 115ms/step - accuracy: 0.1500 - loss: 3.9280 - val_accuracy: 0.1667 - val_loss: 3.7886
60/60 ━━━━━━━━━━━━━━━━━━━━ 20s 93ms/step - accuracy: 0.1667 - loss: 3.8102 - val_accuracy: 0.1667 - val_loss: 3.7730
kNN val acc before=0.0292, after=0.1250


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



===== Flowers =====
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.7575 - loss: 0.7266 - val_accuracy: 0.8175 - val_loss: 0.5268
500/500 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.7816 - loss: 0.6712 - val_accuracy: 0.8425 - val_loss: 0.4769
kNN val acc before=0.7880, after=0.8120


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Saved files in /content/results
['face_test_pca_before_after.png', 'face_test_predictions.csv', 'face_test_samples.png', 'face_test_tsne_before_after.png', 'face_test_umap_before_after.png', 'face_val_pca_before_after.png', 'face_val_tsne_before_after.png', 'face_val_umap_before_after.png', 'flower_test_samples.png', 'flowers_val_pca_before_after.png', 'flowers_val_tsne_before_after.png', 'flowers_val_umap_before_after.png', 'summary_metrics.csv', 'summary_metrics.json']
